# DATA209 — Advanced Exploratory Data Analysis
# Practical P27-28 · PCA — implementation and interpretation

**Vidyashilp University · School of Engineering and Technology**
BTech Hons. (Data Science), Semester III · Week 14 · Module 4 · CO4

---

**Objective.** Implement PCA, visualise the components, and interpret the variance explained and the loadings.

### Dataset

**Online Shoppers Purchasing Intention** — 12,330 browsing sessions × 18 columns
(UCI Machine Learning Repository). One row per session on an e-commerce site over twelve
months; the boolean `Revenue` column marks sessions that ended in a purchase.


### How to run this notebook

This is a **standalone** manual for one 2-hour practical. Run the cells in order:

1. **Setup** — imports and display options. Edit `DATA_DIR` to point at your data folder.
2. **Prepare** — rebuilds the state produced in earlier sessions, so this notebook needs
   nothing from any other file.
3. **The practical** — the session's own work, ending in the deliverable.

> Every figure and every table needs one sentence underneath saying what it shows about the
> problem. A chart without an interpretation earns no marks in this course.

---


## 1 · Setup

Run this first.

In [ ]:
# ============================================================
# DATA209 — Advanced Exploratory Data Analysis
# Lab manual: shared setup. Run this cell first, every session.
# ============================================================
import warnings, os, math, textwrap
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"]  = 110
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.titleweight"] = "bold"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ------------------------------------------------------------------
# EDIT THIS: the folder holding the course CSVs on your machine.
# Keep the data next to this notebook and "." will just work.
# ------------------------------------------------------------------
DATA_DIR = "."

def find(filename, folder=None):
    """Locate a course file, searching DATA_DIR recursively. Returns None if absent."""
    root = folder or DATA_DIR
    direct = os.path.join(root, filename)
    if os.path.exists(direct):
        return direct
    for dirpath, _, files in os.walk(root):
        if filename in files:
            return os.path.join(dirpath, filename)
    return None

print("pandas", pd.__version__, "| numpy", np.__version__)
print("Data folder:", os.path.abspath(DATA_DIR))

## 2 · Prepare

Recap from P1-2 to P23-24, plus the cross-validation splitter used at the end.

In [ ]:
# ------------------------------------------------------------------
# Recap: state built in earlier practicals, rebuilt here so this
# notebook runs on its own. Nothing new is taught in this cell.
# ------------------------------------------------------------------
path = find("online_shoppers_intention.csv")
if path is None:
    raise FileNotFoundError("online_shoppers_intention.csv not found — set DATA_DIR above.")
df = pd.read_csv(path)

TARGET = "Revenue"
y = df[TARGET]
X = df.drop(columns=[TARGET])
numeric_cols     = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

# --- from P15-16
df_clean = df.drop_duplicates().reset_index(drop=True)
for c in ["VisitorType", "Month"]:
    df_clean[c] = df_clean[c].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)

model_cols = ["Administrative", "Administrative_Duration", "Informational",
              "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
              "BounceRates", "ExitRates", "PageValues"]

# --- from P23-24
skew_before = df_clean[model_cols].skew().sort_values(ascending=False)
needs = skew_before[skew_before.abs() > 1].index.tolist()
df_t = df_clean.copy()
for c in needs:
    df_t[c] = np.log1p(df_t[c].clip(lower=0))

# --- validation setup
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print("Ready:", df.shape if "df" in dir() else "media session")

## 3 · P27-28 — PCA — implementation and interpretation

### PCA implementation

PCA finds orthogonal directions of maximum variance. It is **unsupervised** — it never looks at
the target — and it is **scale-sensitive**, so standardising first is not optional.

In [ ]:
# ---- Fit PCA ------------------------------------------------------------
from sklearn.decomposition import PCA

pca_cols = model_cols                  # the transformed original variables
Xp = df_t[pca_cols]

Xp_s = StandardScaler().fit_transform(Xp)
pca  = PCA(random_state=RANDOM_STATE).fit(Xp_s)

evr = pca.explained_variance_ratio_ * 100
variance = pd.DataFrame({
    "component"      : [f"PC{i+1}" for i in range(len(evr))],
    "eigenvalue"     : pca.explained_variance_,
    "variance_%"     : evr,
    "cumulative_%"   : np.cumsum(evr),
}).set_index("component")
print(variance.round(2).to_string())

for target_var in (80, 90, 95):
    k = int(np.argmax(np.cumsum(evr) >= target_var) + 1)
    print(f"Components needed for {target_var}% of variance: {k}")
print("Components with eigenvalue > 1 (Kaiser rule):",
      int((pca.explained_variance_ > 1).sum()))

### Interpret variance explained

In [ ]:
# ---- Scree and cumulative variance --------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))

axes[0].bar(range(1, len(evr) + 1), evr, color="#6B4C7A")
axes[0].plot(range(1, len(evr) + 1), evr, marker="o", color="#B5432E")
axes[0].set_title("Scree plot"); axes[0].set_xlabel("component")
axes[0].set_ylabel("variance explained (%)")

axes[1].plot(range(1, len(evr) + 1), np.cumsum(evr), marker="o", color="#6B4C7A")
for lvl, c in [(80, "#B5432E"), (95, "#8B9199")]:
    axes[1].axhline(lvl, ls="--", lw=1, color=c)
    axes[1].text(0.4, lvl + 1.2, f"{lvl}%", color=c, fontsize=9)
axes[1].set_title("Cumulative variance"); axes[1].set_xlabel("components")
axes[1].set_ylim(0, 105)
plt.tight_layout(); plt.show()

k80 = int(np.argmax(np.cumsum(evr) >= 80) + 1)
print(f"Retention decision: keep {k80} components for 80% of the variance "
      f"({len(pca_cols)} -> {k80}, a {100*(1-k80/len(pca_cols)):.0f}% reduction).")
print("A flat scree means the variables were already fairly independent — that is a finding,")
print("not a failure. Report it rather than forcing a two-component solution.")

### Visualize components

In [ ]:
# ---- Project and plot ---------------------------------------------------
P = pca.transform(Xp_s)
samp = np.random.RandomState(RANDOM_STATE).choice(len(P), 4000, replace=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

sns.scatterplot(x=P[samp, 0], y=P[samp, 1], hue=df_t[TARGET].values[samp],
                palette=["#C9D4DB", "#B5432E"], s=10, alpha=0.55, ax=axes[0])
axes[0].set_title("PC1 vs PC2, coloured by outcome")

sns.scatterplot(x=P[samp, 0], y=P[samp, 2], hue=df_t[TARGET].values[samp],
                palette=["#C9D4DB", "#B5432E"], s=10, alpha=0.55, ax=axes[1], legend=False)
axes[1].set_title("PC1 vs PC3")

# biplot: how each original variable loads onto the first two components
load = pca.components_[:2].T * np.sqrt(pca.explained_variance_[:2])
for i, name in enumerate(pca_cols):
    axes[2].arrow(0, 0, load[i, 0], load[i, 1], head_width=0.03,
                  color="#6B4C7A", alpha=0.8)
    axes[2].text(load[i, 0] * 1.12, load[i, 1] * 1.12, name, fontsize=7, ha="center")
axes[2].set_xlim(-1.2, 1.2); axes[2].set_ylim(-1.2, 1.2)
axes[2].axhline(0, lw=.6, color="grey"); axes[2].axvline(0, lw=.6, color="grey")
axes[2].set_title("Loading biplot (PC1 / PC2)")

for a in axes[:2]:
    a.set_xlabel(f"PC1 ({evr[0]:.1f}%)"); a.set_ylabel(f"PC2 ({evr[1]:.1f}%)")
plt.tight_layout(); plt.show()

In [ ]:
# ---- Loadings: what is each component actually made of? ----------------
loadings = pd.DataFrame(
    pca.components_[:4].T,
    columns=[f"PC{i+1}" for i in range(4)],
    index=pca_cols)

print("Loadings (first four components)")
print(loadings.round(3).to_string())

print("\nDominant variables per component")
for pc in loadings.columns:
    top = loadings[pc].abs().sort_values(ascending=False).head(3)
    parts = [f"{name} ({loadings.loc[name, pc]:+.2f})" for name in top.index]
    print(f"  {pc} ({evr[int(pc[2:])-1]:.1f}% var): " + ", ".join(parts))

plt.figure(figsize=(8, 3.6))
sns.heatmap(loadings, annot=True, fmt=".2f", center=0, cmap="PuOr",
            cbar_kws={"shrink": .8}, annot_kws={"size": 8})
plt.title("Component loadings"); plt.tight_layout(); plt.show()

print("\nNow NAME each component from its loadings — 'session depth', 'bounce/exit behaviour',")
print("'checkout intent'. A component you cannot name should not appear in your report.")

In [ ]:
# ---- Is the reduction worth it? ----------------------------------------
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipe_full = Pipeline([("scale", StandardScaler()),
                      ("model", LogisticRegression(max_iter=2000,
                                                   random_state=RANDOM_STATE))])
pipe_pca  = Pipeline([("scale", StandardScaler()),
                      ("pca", PCA(n_components=k80, random_state=RANDOM_STATE)),
                      ("model", LogisticRegression(max_iter=2000,
                                                   random_state=RANDOM_STATE))])

yv = df_t[TARGET].astype(int)
s_full = cross_val_score(pipe_full, Xp, yv, cv=cv, scoring="roc_auc", n_jobs=-1)
s_pca  = cross_val_score(pipe_pca,  Xp, yv, cv=cv, scoring="roc_auc", n_jobs=-1)

print(pd.DataFrame({
    "pipeline"  : [f"all {len(pca_cols)} variables", f"PCA -> {k80} components"],
    "roc_auc"   : [s_full.mean(), s_pca.mean()],
    "std"       : [s_full.std(), s_pca.std()],
}).round(4).to_string(index=False))

print("\nNote that PCA sits INSIDE the pipeline, so it is refitted on each training fold.")
print("Fitting PCA on the whole dataset before cross-validation would leak test information.")

### Deliverable — P27-28

A notebook containing the variance table, scree and cumulative plots, the retention decision with
its justification, the 2-D projections, the loading heatmap, and a **plain-language name for every
retained component**.